# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. "A": 2 and "B": 1. Потому что метод get ищет ключ в словаре, и если он находит, то возвращает ее значение и прибавляет 1, а если нет, то возвращает 0 и также прибавляет 1 и сохраняет ключ с значением в словарь
2. [0, 1, 1]. Потому что метод append каждый раз добавляет новый элемент в конец листа, и на примере мы вызвали функцию 2 раза, тем самым добавив два элемента в конец
3. 2. Потому что создается множество, а множество сохраняет только уникальные элементы, и когда каждое название города переводится на строчный, дубликат almaty удаляется и тем самым длина получается 2
4. "ab": 2, "abc": 3. На этот раз создается словарь, и в словаре ключи должны быть уникальными, и когда мы проходим по циклу, он не создает отдельный ключ для второго ab
5. 5, None. Потому что print просто выводит значение в терминал а не возвращает, и когда вывелся 5, то в x это значение никак не передалось, и x выводит None из за того что значения там нет

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# База

In [6]:
#1
def to_num(s):
  s = s.replace(" ", "").strip()
  return float(s)

def clean_city(s):
  cities = {
    "almaty": "Алматы",
    "astana": "Астана",
    "karaganda": "Караганда",
    "shymkent": "Шымкент",
    "atyrau": "Атырау",
    "aktobe": "Актобе"
  }
  return cities.get(s.strip().lower(), s.strip().capitalize())

In [7]:
#2
counts = {}

for row in rows:
    category = row["category"]
    counts[category] = counts.get(category, 0) + 1

for category, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(category, count)

Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653


In [8]:
#3
revenue = {}
counts = {}

for row in rows:
    if row["status"] == "оплачен":
        category = row["category"]
        amount = to_num(row["amount"])

        revenue[category] = revenue.get(category, 0) + amount
        counts[category] = counts.get(category, 0) + 1

print(f"{'Категория':<15} {'Выручка':>15} {'Средний чек':>15}")

for category in revenue:
    average = revenue[category] / counts[category]
    print(f"{category:<15} {revenue[category]:>15.2f} {average:>15.2f}")

Категория               Выручка     Средний чек
Одежда              18321080.00        13285.77
Красота              5507620.00         6188.34
Дом                 10516320.00        10654.83
Книги                2294100.00         3934.99
Электроника         65270240.00        61286.61


In [10]:
#4
counts = {}

for row in rows:
    month = row["order_date"][:7]
    counts[month] = counts.get(month, 0) + 1

for month, count in sorted(counts.items()):
    print(f"{month}: {count}")

2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт

In [23]:
#5
revenue = {}

for row in rows:
    if row["status"] == "оплачен":
        month = row["order_date"][:7]
        amount = to_num(row["amount"])
        revenue[month] = revenue.get(month, 0) + amount
for month, total in sorted(revenue.items()):
    print(f"{month}: {total:.2f}")

september = revenue.get("2026-09", 0)
october = revenue.get("2026-10", 0)

if september != 0:
    change = (october - september) / september * 100
    print(f"изменение октября к сентябрю: {change:+.2f}%")
else:
    print("гельзя рассчитать изменение: выручка сентября равна нулю")

2026-09: 57489850.00
2026-10: 44419510.00
изменение октября к сентябрю: -22.74%


In [24]:
#6
total = {}
cancelled = {}

for row in rows:
    city = clean_city(row["city"])
    total[city] = total.get(city, 0) + 1

    if row["status"] == "отменён":
        cancelled[city] = cancelled.get(city, 0) + 1

for city in sorted(total):
    percent = cancelled.get(city, 0) / total[city] * 100
    print(f"{city}: {percent:.1f}%")

Актобе: 10.3%
Алматы: 7.9%
Астана: 8.8%
Атырау: 7.5%
Караганда: 8.9%
Шымкент: 20.4%


In [25]:
#7
revenue = {}

for row in rows:
    if row["status"] == "оплачен":
        customer = row["customer_id"]
        revenue[customer] = revenue.get(customer, 0) + to_num(row["amount"])

top5 = sorted(revenue.items(), key=lambda x: x[1], reverse=True)[:5]
for customer, total in top5:
    print(f"{customer}: {total:.2f}")

TEST: 29999970.00
C01627: 257180.00
C00485: 215920.00
C01338: 192080.00
C02122: 191670.00


На первом месте TEST. У него три оплаченных заказа по 9 999 990 каждый. Само название говорит что это тестовые данные, и они искажают выручку

In [26]:
#8
clients = {}

for row in rows:
    city = clean_city(row["city"])
    customer = row["customer_id"]

    if city not in clients:
        clients[city] = set()

    clients[city].add(customer)

for city, customers in sorted(clients.items()):
    print(f"{city}: {len(customers)}")

Актобе: 358
Алматы: 1351
Астана: 981
Атырау: 138
Караганда: 451
Шымкент: 655


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж